💡 **Environment:** `clamp-analyses`

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
  library(patchwork)
  library(cowplot)
  library(yaml)
  library(here)
  library(grid)
  library(ragg)
  library(svglite)
})


In [ ]:
FONT_FAMILY <- "Helvetica"

FS_TAG          <- 9
FS_TITLE        <- 8
FS_SUBTITLE     <- 6.5
FS_AXIS_TITLE   <- 7
FS_AXIS_TEXT    <- 6.5
FS_LEGEND       <- 5.5
FS_LEGEND_TITLE <- 6
FS_STAT         <- 6
FS_HEAT_LABEL   <- 5.5
FS_HEAT_VALUE   <- 5.5
FS_CELL_VALUE   <- 5
FS_DENSE_VALUE  <- 4
FS_A_MEAN       <- 4
FS_HEAT_LABEL_H <- 5

LINE_W <- 0.3
GRID_W <- 0.2

pt_mm <- function(pt) pt / .pt

theme_nature_methods <- function(base_size = FS_AXIS_TEXT,
                                 grid = c("none", "y", "x", "both")) {
  grid <- match.arg(grid)
  th <- theme_classic(base_size = base_size, base_family = FONT_FAMILY) %+replace%
    theme(
      plot.title        = element_text(size = FS_TITLE, face = "bold", hjust = 0.5,
                                       margin = margin(b = 1)),
      plot.subtitle     = element_text(size = FS_SUBTITLE, face = "plain", hjust = 0.5,
                                       lineheight = 0.95, margin = margin(b = 1)),
      plot.tag          = element_text(size = FS_TAG, face = "bold", family = FONT_FAMILY),
      plot.tag.position = "topleft",
      plot.tag.location = "margin",
      axis.line         = element_line(linewidth = LINE_W, colour = "black"),
      axis.ticks        = element_line(linewidth = LINE_W, colour = "black"),
      axis.ticks.length = unit(0.9, "pt"),
      axis.text         = element_text(size = base_size, colour = "black"),
      axis.text.x       = element_text(margin = margin(t = 0.8)),
      axis.text.y       = element_text(hjust = 1, margin = margin(r = 0.8)),
      axis.title        = element_text(size = FS_AXIS_TITLE, colour = "black"),
      axis.title.x      = element_text(margin = margin(t = 1)),
      axis.title.y      = element_text(angle = 90, margin = margin(r = 1)),
      legend.text       = element_text(size = FS_LEGEND),
      legend.title      = element_text(size = FS_LEGEND_TITLE),
      legend.key.size   = unit(2, "mm"),
      legend.key        = element_blank(),
      legend.background = element_blank(),
      legend.margin     = margin(0, 0, 0, 0),
      legend.box.margin = margin(0, 0, 0, 0),
      strip.text        = element_text(size = FS_TITLE, face = "bold", margin = margin(b = 1)),
      strip.background  = element_blank(),
      panel.background  = element_blank(),
      panel.grid        = element_blank(),
      plot.background   = element_blank(),
      plot.margin       = margin(1, 1, 1, 1, "mm")
    )
  if (grid %in% c("y", "both"))
    th <- th + theme(panel.grid.major.y = element_line(colour = "grey88", linewidth = GRID_W))
  if (grid %in% c("x", "both"))
    th <- th + theme(panel.grid.major.x = element_line(colour = "grey88", linewidth = GRID_W))
  th
}

theme_nature_heatmap <- function(x_angle = 45, label_size = FS_HEAT_LABEL) {
  theme_nature_methods() %+replace%
    theme(
      axis.line   = element_blank(),
      axis.ticks  = element_blank(),
      axis.text.x = element_text(size = label_size, angle = x_angle,
                                 hjust = if (x_angle == 0) 0.5 else 1,
                                 vjust = if (x_angle == 90) 0.5 else 1,
                                 colour = "black",
                                 margin = margin(t = 0.5), lineheight = 0.9),
      axis.text.y = element_text(size = label_size, hjust = 1, colour = "black",
                                 margin = margin(r = 0.5), lineheight = 0.9),
      plot.margin = margin(1, 1, 1, 1, "mm")
    )
}

add_tag <- function(p, tag) {
  p + labs(tag = tag) +
    theme(plot.tag          = element_text(size = FS_TAG, face = "bold",
                                           family = FONT_FAMILY),
          plot.tag.position = "topleft",
          plot.tag.location = "margin")
}

add_overlay_tag <- function(p, tag) {
  tagged <- ggdraw(p) +
    draw_label(tag, x = 0, y = 1, hjust = 0, vjust = 1,
               size = FS_TAG, fontface = "bold", fontfamily = FONT_FAMILY)
  wrap_elements(full = tagged)
}

DATASET_LABELS <- c(
  Brain_Mathys2023 = "Brain Mathys",
  Brain_Xiong2023  = "Brain Xiong",
  Heart_Datar2026  = "Heart Datar",
  PBMC_1k1k        = "PBMC 1k1k",
  PBMC_Perez2022   = "PBMC Perez",
  Lung_Sikkema2023 = "Lung Sikkema"
)

TISSUE_MAP <- c(
  Brain_Mathys2023 = "Brain", Brain_Xiong2023 = "Brain",
  Heart_Datar2026  = "Heart",
  Lung_Sikkema2023 = "Lung",
  PBMC_1k1k        = "PBMC", PBMC_Perez2022 = "PBMC"
)

DATASETS_ROW1 <- c("Heart_Datar2026", "PBMC_1k1k", "Lung_Sikkema2023")
DATASETS_ROW2 <- c("Brain_Mathys2023", "Brain_Xiong2023", "PBMC_Perez2022")


TISSUE_ORDER <- c("Brain", "Heart", "Lung", "PBMC")

abbreviate_ct <- function(x) {
  x <- gsub("Oligodendrocyte [Pp]rogenitor [Cc]ells?", "OPC", x)
  x <- gsub("Oligodendrocyte [Pp]recursor [Cc]ells?", "OPC", x)
  x
}

CT_SHORT <- c(
  "Oligodendrocyte Precursor Cells" = "OPC",
  "Plasmacytoid Dendritic Cells"    = "pDC",
  "LymphaticEndothelial"            = "Lymphatic EC",
  "Alveolar epithelium"             = "Alveolar ep.",
  "Airway epithelium"               = "Airway ep.",
  "Excitatory neurons"              = "Excitatory",
  "Inhibitory neurons"              = "Inhibitory",
  "Endothelial cells"               = "Endothelial",
  "Fibroblast lineage"              = "Fibroblast",
  "Submucosal Gland"                = "Submucosal",
  "Oligodendrocytes"                = "Oligodendro.",
  "CD14+ Monocytes"                 = "CD14+ Mono",
  "CD16+ Monocytes"                 = "CD16+ Mono",
  "Dendritic cells"                 = "Dendritic",
  "Plasma B cells"                  = "Plasma B",
  "Myeloid cells"                   = "Myeloid",
  "Malignant cells"                 = "Malignant",
  "Vascular cells"                  = "Vascular",
  "Cancer cells"                    = "Cancer",
  "Mast cells"                      = "Mast",
  "Blood vessels"                   = "Blood vessel"
)
short_ct <- function(x) {
  x <- abbreviate_ct(x)
  ifelse(x %in% names(CT_SHORT), CT_SHORT[x], x)
}

wrap_label <- function(x, width = 14) {
  vapply(x, function(s) paste(strwrap(s, width = width), collapse = "\n"),
         character(1), USE.NAMES = FALSE)
}

fmt_corr_cell <- function(x) sub("0.", ".", sprintf("%.2f", x), fixed = TRUE)

shorten_gtex <- function(x) {
  x <- gsub("Adipose - Subcutaneous",                    "Adipose - Subcut.", x)
  x <- gsub("Adipose - Visceral \\(Omentum\\)",             "Adipose - Visceral", x)
  x <- gsub("Artery - ",                                 "Artery - ", x)
  x <- gsub("Brain - Amygdala",                          "Brain - Amygdala", x)
  x <- gsub("Brain - Anterior cingulate cortex \\(BA24\\)", "Brain - ACC (BA24)", x)
  x <- gsub("Brain - Caudate \\(basal ganglia\\)",         "Brain - Caudate", x)
  x <- gsub("Brain - Cerebellar Hemisphere",             "Brain - Cereb. hem.", x)
  x <- gsub("Brain - Frontal Cortex \\(BA9\\)",            "Brain - FC (BA9)", x)
  x <- gsub("Brain - Nucleus accumbens \\(basal ganglia\\)", "Brain - NAc", x)
  x <- gsub("Brain - Putamen \\(basal ganglia\\)",         "Brain - Putamen", x)
  x <- gsub("Brain - Spinal cord \\(cervical c-1\\)",      "Brain - Spinal cord", x)
  x <- gsub("Brain - Substantia nigra",                  "Brain - Subst. nigra", x)
  x <- gsub("Breast - Mammary Tissue",                   "Breast - Mammary", x)
  x <- gsub("Cells - Cultured fibroblasts",              "Cells - Fibroblasts", x)
  x <- gsub("Cells - EBV-transformed lymphocytes",       "Cells - EBV lymph.", x)
  x <- gsub("Esophagus - Gastroesophageal Junction",     "Esoph. - GE junction", x)
  x <- gsub("Esophagus - Mucosa",                        "Esoph. - Mucosa", x)
  x <- gsub("Esophagus - Muscularis",                    "Esoph. - Muscularis", x)
  x <- gsub("Heart - Atrial Appendage",                  "Heart - Atrial app.", x)
  x <- gsub("Minor Salivary Gland",                      "Minor saliv. gland", x)
  x <- gsub("Skin - Not Sun Exposed \\(Suprapubic\\)",     "Skin - Not sun exp.", x)
  x <- gsub("Skin - Sun Exposed \\(Lower leg\\)",          "Skin - Sun exp.", x)
  x <- gsub("Small Intestine - Terminal Ileum",          "Sm. intestine - Ileum", x)
  trimws(x)
}

fmt_q_compact <- function(q) {
  if (is.na(q)) return('"n.a."')
  if (q >= 0.001) return(sprintf('"%.3f"', q))
  e_str    <- formatC(q, format = "e", digits = 1)
  parts    <- strsplit(e_str, "e")[[1]]
  sprintf('%s%%*%%10^{%d}', trimws(parts[1]), as.integer(parts[2]))
}

assign_bracket_tiers <- function(comp_df, xpos) {
  comp_ord <- copy(as.data.table(comp_df))
  comp_ord[, x1 := xpos[a]]
  comp_ord[, x2 := xpos[b]]
  comp_ord[, left  := pmin(x1, x2)]
  comp_ord[, right := pmax(x1, x2)]
  comp_ord[, span  := right - left]
  setorder(comp_ord, span, q)

  levels_used <- list()
  comp_ord[, tier := 0L]
  for (i in seq_len(nrow(comp_ord))) {
    left  <- comp_ord$left[i]; right <- comp_ord$right[i]; tier <- 1L
    repeat {
      current  <- if (tier <= length(levels_used)) levels_used[[tier]] else NULL
      overlaps <- !is.null(current) && any(vapply(current, function(iv) {
        !(right < iv[1] || left > iv[2])
      }, logical(1)))
      if (!overlaps) break
      tier <- tier + 1L
    }
    comp_ord$tier[i] <- tier
    prior <- if (tier <= length(levels_used)) levels_used[[tier]] else list()
    levels_used[[tier]] <- c(prior, list(c(left, right)))
  }
  comp_ord
}

add_brackets <- function(p, comp_ord, y_base, y_step, h, size = pt_mm(FS_STAT)) {
  for (i in seq_len(nrow(comp_ord))) {
    r <- comp_ord[i, ]
    y <- y_base + (r$tier - 1) * y_step
    p <- p +
      annotate("segment", x = r$left,  xend = r$left,  y = y,     yend = y + h, linewidth = 0.2) +
      annotate("segment", x = r$left,  xend = r$right, y = y + h, yend = y + h, linewidth = 0.2) +
      annotate("segment", x = r$right, xend = r$right, y = y,     yend = y + h, linewidth = 0.2) +
      annotate("text", x = (r$left + r$right) / 2, y = y + h + 0.005,
               label = fmt_q_compact(r$q), size = size, vjust = 0,
               parse = TRUE, family = FONT_FAMILY)
  }
  p
}

cfg <- yaml::read_yaml(here("config.yaml"))
MODEL_COLORS_RAW <- unlist(cfg$MODEL_COLORS)
names(MODEL_COLORS_RAW)[names(MODEL_COLORS_RAW) == "GenomicSuperSignature"] <- "GSSig"

ct_labels_df <- read.csv(here("data", "pseudobulk", "cell_type_labels.csv"), stringsAsFactors = FALSE)
CT_LABELS <- setNames(ct_labels_df$label, ct_labels_df$cell_type)
ct_label <- function(x) ifelse(x %in% names(CT_LABELS), CT_LABELS[x], x)

GREEN_SCALE      <- unlist(cfg$GREEN_SCALE)
DIVERGING_COLORS <- unlist(cfg$DIVERGING_COLORS)
DIVERGING_VALUES <- as.numeric(cfg$DIVERGING_VALUES)

TILE_TEXT <- c(`TRUE` = "white", `FALSE` = "black")


## Panel A: benchmark boxplots, faceted by tissue

In [ ]:
long <- fread(snakemake@input[["benchmark_long"]])
stopifnot(all(c("dataset", "method", "truth", "cor") %in% names(long)))

long_box <- long[truth == "v0" & !is.na(cor)]
long_box[, tissue := TISSUE_MAP[dataset]]
stopifnot(!anyNA(long_box$tissue))

METHOD_ORDER  <- long_box[, .(m = mean(cor, na.rm = TRUE)), by = method][order(-m), as.character(method)]
METHOD_COLORS <- MODEL_COLORS_RAW[METHOD_ORDER]
METHOD_COLORS[is.na(METHOD_COLORS)] <- "grey70"
names(METHOD_COLORS) <- METHOD_ORDER

long_box[, method := factor(method, levels = METHOD_ORDER)]
mean_by_tissue <- long_box[, .(mean_cor = mean(cor, na.rm = TRUE),
                               max_cor  = max(cor, na.rm = TRUE)), by = .(tissue, method)]

A_Y_MAX <- 1.16

make_tissue_box <- function(tis, show_x, show_y) {
  d <- long_box[tissue == tis]
  m <- mean_by_tissue[tissue == tis]
  p <- ggplot(d, aes(method, cor, fill = method)) +
    geom_boxplot(width = 0.6, outlier.shape = NA, colour = "black",
                 linewidth = LINE_W, alpha = 0.85) +
    geom_jitter(width = 0.10, size = 0.3, shape = 21, fill = "white",
                colour = "#333333", stroke = 0.12, alpha = 0.7) +
    geom_point(data = m, aes(x = method, y = mean_cor), shape = 23, size = 1.1,
               fill = "white", colour = "black", stroke = 0.3, inherit.aes = FALSE) +
    geom_text(data = m, aes(x = method, y = 1.05, label = sprintf("%.3f", mean_cor)),
              size = pt_mm(FS_A_MEAN), colour = "black", inherit.aes = FALSE) +
    scale_x_discrete(drop = FALSE) +
    scale_y_continuous(breaks = seq(0, 1, 0.25),
                       labels = c("0", "0.25", "0.5", "0.75", "1.0"),
                       expand = expansion(mult = c(0.02, 0.02))) +
    coord_cartesian(ylim = c(0, A_Y_MAX), clip = "on") +
    scale_fill_manual(values = METHOD_COLORS, na.value = "grey70", drop = FALSE) +
    labs(x = NULL, y = "Max Pearson r per cell type", title = tis) +
    theme_nature_methods(grid = "none") +
    theme(legend.position = "none")
  p <- if (show_x) {
    p + theme(axis.text.x = element_text(angle = 35, hjust = 1, vjust = 1,
                                         size = FS_AXIS_TEXT, margin = margin(t = 0.8)))
  } else {
    p + theme(axis.text.x = element_blank(), axis.ticks.x = element_blank())
  }
  if (!show_y) p <- p + theme(axis.text.y = element_blank(), axis.ticks.y = element_blank())
  p
}

plot_A_brain <- make_tissue_box("Brain", show_x = FALSE, show_y = TRUE)
plot_A_heart <- make_tissue_box("Heart", show_x = FALSE, show_y = FALSE)
plot_A_lung  <- make_tissue_box("Lung",  show_x = TRUE,  show_y = TRUE)
plot_A_pbmc  <- make_tissue_box("PBMC",  show_x = TRUE,  show_y = FALSE)

A_grid <- wrap_plots(plot_A_brain, plot_A_heart, plot_A_lung, plot_A_pbmc,
                     ncol = 2, heights = c(1, 1)) +
  plot_layout(axis_titles = "collect_y")

panel_A <- add_overlay_tag(A_grid, "A")

options(repr.plot.width = 6, repr.plot.height = 3)
print(panel_A)


## Panel B: bootstrap win rate

In [ ]:
win_rate <- fread(snakemake@input[["bootstrap"]])
stopifnot(all(c("method", "win_rate") %in% names(win_rate)))

setorder(win_rate, win_rate)
win_rate[, method := factor(method, levels = method)]

panel_B <- ggplot(win_rate, aes(x = win_rate, y = method, fill = method)) +
  geom_col(width = 0.72) +
  geom_text(aes(label = sprintf("%.0f%%", 100 * win_rate)),
            hjust = -0.2, size = pt_mm(FS_AXIS_TEXT), colour = "black") +
  scale_fill_manual(values = MODEL_COLORS_RAW, na.value = "grey70") +
  scale_x_continuous(limits = c(0, 1), breaks = seq(0, 1, 0.25),
                     labels = function(x) paste0(100 * x, "%"),
                     expand = expansion(mult = c(0, 0.22))) +
  labs(x = NULL, y = NULL) +
  theme_nature_methods(grid = "none") +
  theme(legend.position = "none",
        axis.line.y   = element_blank(),
        axis.ticks.y  = element_blank(),
        axis.text.x   = element_text(margin = margin(t = 0)),
        axis.ticks.length.x = unit(0.6, "pt"),
        plot.margin   = margin(1, 1, 0.3, 1, "mm"))

panel_B <- add_tag(panel_B, "B")

options(repr.plot.width = 2, repr.plot.height = 3)
print(panel_B)


## Panel C: per-dataset single-cell recovery (purity) heatmaps

In [ ]:
heatmap_long <- fread(snakemake@input[["heatmap_long"]])
stopifnot(all(c("dataset", "row_cell_type", "col_cell_type", "pct") %in% names(heatmap_long)))

PURITY_LABEL_MIN <- 15

make_purity_panel <- function(ds) {
  d <- heatmap_long[dataset == ds]
  stopifnot(nrow(d) > 0)

  ct_order  <- unique(d$row_cell_type)
  ct_order  <- ct_order[order(ct_label(ct_order))]
  labs_ord  <- short_ct(ct_label(ct_order))

  d[, row_label   := factor(short_ct(ct_label(row_cell_type)), levels = labs_ord)]
  d[, col_label   := factor(short_ct(ct_label(col_cell_type)), levels = labs_ord)]
  d[, is_diagonal := row_cell_type == col_cell_type]
  d[, show_label  := is_diagonal | pct >= PURITY_LABEL_MIN]

  ggplot(d, aes(x = col_label, y = row_label, fill = pct)) +
    geom_tile(colour = "white", linewidth = 0.25) +
    geom_tile(data = d[is_diagonal == TRUE], fill = NA, colour = "black", linewidth = 0.4) +
    geom_text(data = d[show_label == TRUE],
              aes(label = sprintf("%.0f", pct), colour = pct >= 65),
              size = pt_mm(FS_CELL_VALUE), show.legend = FALSE) +
    scale_fill_gradientn(colours = GREEN_SCALE, limits = c(0, 100),
                         name = "Top 1% purity (%)",
                         guide = guide_colourbar(barwidth = unit(20, "mm"),
                                                 barheight = unit(1.8, "mm"),
                                                 title.position = "left",
                                                 title.vjust = 1)) +
    scale_colour_manual(values = TILE_TEXT, guide = "none") +
    scale_x_discrete(expand = c(0, 0)) +
    scale_y_discrete(expand = c(0, 0)) +
    labs(x = NULL, y = NULL, title = DATASET_LABELS[ds]) +
    theme_nature_heatmap(x_angle = 45, label_size = FS_HEAT_LABEL_H)
}

C_row1_panels <- lapply(DATASETS_ROW1, make_purity_panel)
C_row2_panels <- lapply(DATASETS_ROW2, make_purity_panel)

C_N1 <- vapply(DATASETS_ROW1, function(ds) uniqueN(heatmap_long[dataset == ds, row_cell_type]), numeric(1))
C_N2 <- vapply(DATASETS_ROW2, function(ds) uniqueN(heatmap_long[dataset == ds, row_cell_type]), numeric(1))

C_LEG <- max(C_N2)

C_row1 <- wrap_plots(C_row1_panels, nrow = 1, widths = C_N1) &
  theme(legend.position = "none")
C_row2 <- wrap_plots(c(C_row2_panels, list(guide_area())),
                     nrow = 1, widths = c(C_N2, C_LEG),
                     guides = "collect") &
  theme(legend.position = "bottom", legend.direction = "horizontal",
        legend.justification = "left")

panel_C_core <- wrap_plots(C_row1, C_row2, ncol = 1, heights = c(1.35, 1))
panel_C <- wrap_elements(full =
  ggdraw() +
    draw_plot(panel_C_core, x = 0.042, y = 0.038, width = 0.958, height = 0.962) +
    draw_label("LV assigned to cell type", x = 0.52, y = 0.002,
               hjust = 0.5, vjust = 0, size = FS_AXIS_TITLE,
               fontfamily = FONT_FAMILY) +
    draw_label("Cell type of top 1% projected cells", x = 0.006, y = 0.52,
               angle = 90, hjust = 0.5, vjust = 0, size = FS_AXIS_TITLE,
               fontfamily = FONT_FAMILY) +
    draw_label("C", x = 0, y = 1, hjust = 0, vjust = 1,
               size = FS_TAG, fontface = "bold", fontfamily = FONT_FAMILY)
)

options(repr.plot.width = 7.2, repr.plot.height = 2.5)
print(panel_C)


## Panel D: per-dataset LV x cell-type correlation heatmaps

In [ ]:
corr_full          <- fread(snakemake@input[["corr_full"]])
assignments        <- fread(snakemake@input[["assignments"]])
stopifnot(all(c("dataset", "LV", "cell_type", "cor") %in% names(corr_full)))
stopifnot(all(c("dataset", "LV", "cell_type") %in% names(assignments)))

CORR_LABEL_MIN <- 0.6

make_corr_panel <- function(ds) {
  d        <- corr_full[dataset == ds & !is.na(cell_type) & cell_type != "NA"]
  assigned <- assignments[dataset == ds & !is.na(cell_type) & cell_type != "NA"]
  stopifnot(nrow(d) > 0, nrow(assigned) > 0)
  d <- d[LV %in% assigned$LV & cell_type %in% assigned$cell_type]

  ord      <- order(assigned$cell_type)
  lv_order <- unique(assigned$LV[ord])
  ct_order <- unique(short_ct(ct_label(assigned$cell_type[ord])))

  assigned_key <- paste(assigned$LV, assigned$cell_type)
  d[, cell_type_label := factor(short_ct(ct_label(cell_type)), levels = ct_order)]
  d[, LV              := factor(LV, levels = lv_order)]
  d[, is_assigned     := paste(LV, cell_type) %in% assigned_key]
  d[, show_label      := is_assigned | abs(cor) >= CORR_LABEL_MIN]
  value_size <- if (ds %in% DATASETS_ROW1) FS_DENSE_VALUE else FS_CELL_VALUE

  ggplot(d, aes(x = LV, y = cell_type_label, fill = cor)) +
    geom_tile(colour = "white", linewidth = 0.25) +
    geom_tile(data = d[is_assigned == TRUE], fill = NA, colour = "black", linewidth = 0.4) +
    geom_text(data = d[show_label == TRUE],
              aes(label = fmt_corr_cell(cor), colour = abs(cor) >= 0.8),
              size = pt_mm(value_size), show.legend = FALSE) +
    scale_fill_gradientn(colours = DIVERGING_COLORS, values = DIVERGING_VALUES,
                         limits = c(-1, 1), name = "Pearson r",
                         guide = guide_colourbar(barwidth = unit(20, "mm"),
                                                 barheight = unit(1.8, "mm"),
                                                 title.position = "left",
                                                 title.vjust = 1)) +
    scale_colour_manual(values = TILE_TEXT, guide = "none") +
    scale_x_discrete(expand = c(0, 0)) +
    scale_y_discrete(expand = c(0, 0)) +
    labs(x = NULL, y = NULL, title = DATASET_LABELS[ds]) +
    theme_nature_heatmap(x_angle = 45, label_size = FS_HEAT_LABEL_H) +
    theme(plot.title    = element_text(size = FS_SUBTITLE + 0.5, face = "bold",
                                       hjust = 0.5, margin = margin(b = 0.5)),
          plot.subtitle = element_blank())
}

D_row1_panels <- lapply(DATASETS_ROW1, make_corr_panel)
D_row2_panels <- lapply(DATASETS_ROW2, make_corr_panel)
D_row1_panels[[1]] <- add_tag(D_row1_panels[[1]], "D")

D_ncat <- function(ds) uniqueN(assignments[dataset == ds & !is.na(cell_type) & cell_type != "NA", LV])
D_N1 <- vapply(DATASETS_ROW1, D_ncat, numeric(1))
D_N2 <- vapply(DATASETS_ROW2, D_ncat, numeric(1))
D_row1 <- wrap_plots(D_row1_panels, nrow = 1, widths = D_N1)
D_row2 <- wrap_plots(D_row2_panels, nrow = 1, widths = D_N2)

panel_D <- wrap_plots(D_row1, D_row2, ncol = 1, heights = c(1.35, 1))

options(repr.plot.width = 5, repr.plot.height = 2.5)
print(panel_D)


## Panel E: marker-recovery dot grid, by dataset

Demoted from the main figure. For every dataset, each cell type's assigned latent
variable against the marker sets, sized by FDR and coloured by the LV's effect.

In [ ]:
module_ready <- fread(snakemake@input[["marker"]])
GRID_COLORS <- unlist(cfg$GRID_COLORS)
GRID_VALUES <- as.numeric(cfg$GRID_VALUES)

build_module_pub <- function(d) {
  ds <- d$dataset[1]
  ds_label <- if (ds %in% names(DATASET_LABELS)) DATASET_LABELS[ds] else ds
  d[, marker_row_label := abbreviate_ct(marker_row_label)]
  d[, lv_col_label := wrap_label(lv_col_label)]
  row_levels <- unique(d[order(marker_row_rank), marker_row_label])
  col_levels <- unique(d[order(lv_col_rank), lv_col_label])
  d[, marker_row_label := factor(marker_row_label, levels = row_levels)]
  d[, lv_col_label := factor(lv_col_label, levels = col_levels)]
  diagonal <- d[diag_recovered == TRUE]
  z_lim <- d$Z_LIM[1]
  q_lim <- d$Q_LIM[1]
  n_samples <- d$n_samples[1]

  p <- ggplot(d, aes(lv_col_label, marker_row_label)) +
    geom_point(aes(size = neg_log10_fdr, color = row_effect)) +
    {if (nrow(diagonal)) geom_point(data = diagonal, aes(size = neg_log10_fdr),
                                    shape = 1, color = "black", stroke = 0.55)} +
    scale_size_continuous(limits = c(0, q_lim), range = c(0.25, 2.15),
                          name = expression("-" * log[10] ~ "(FDR)")) +
    scale_color_gradientn(colors = GRID_COLORS, values = GRID_VALUES,
                          limits = c(-z_lim, z_lim), name = "LV effect") +
    scale_x_discrete(drop = FALSE) +
    scale_y_discrete(drop = FALSE) +
    coord_fixed() +
    labs(x = NULL, y = NULL, title = paste0(ds_label, " (n = ", n_samples, ")")) +
    theme_nature_methods() +
    theme(panel.grid.major = element_line(color = "grey90", linewidth = 0.18),
          axis.line = element_blank(), axis.ticks = element_blank(),
          axis.text.x = element_text(angle = 45, hjust = 1, vjust = 1,
                                     size = 4.3, lineheight = 0.88),
          axis.text.y = element_text(size = 4.7),
          plot.title = element_text(size = 5.7, face = "bold", hjust = 0.5,
                                    margin = margin(b = 0.5)),
          plot.margin = margin(0.5, 0.7, 0.5, 0.7))
  list(plot = p, n_cat = length(row_levels))
}

module_pub_blocks <- lapply(split(copy(module_ready), module_ready$dataset), build_module_pub)
module_pub_order <- order(-vapply(module_pub_blocks, `[[`, "n_cat", FUN.VALUE = numeric(1)))
module_pub_blocks <- module_pub_blocks[module_pub_order]
module_pub_panels <- lapply(module_pub_blocks, `[[`, "plot")
module_pub_widths <- sqrt(vapply(module_pub_blocks, `[[`, "n_cat", FUN.VALUE = numeric(1)))

top_idx <- 1:3
bottom_idx <- 4:6
top_sum <- sum(module_pub_widths[top_idx])
bottom_sum <- sum(module_pub_widths[bottom_idx])
legend_pad <- max(0.8, top_sum - bottom_sum)

module_legend <- get_legend(
  module_pub_panels[[1]] +
    guides(color = guide_colorbar(order = 1, title.position = "top",
                                  barwidth = unit(16, "mm"),
                                  barheight = unit(1.5, "mm")),
           size = guide_legend(order = 2, title.position = "top", nrow = 1)) +
    theme(legend.position = "bottom", legend.direction = "horizontal",
          legend.box = "vertical", legend.box.just = "center",
          legend.spacing.y = unit(0.5, "mm"),
          legend.box.margin = margin(0, 0, 0, 0),
          legend.key.width = unit(2.1, "mm"), legend.key.height = unit(1.8, "mm"),
          legend.text = element_text(size = 4.7),
          legend.title = element_text(size = 5))
)

C_top <- plot_grid(plotlist = lapply(module_pub_panels[top_idx],
                                     function(p) p + theme(legend.position = "none")),
                   nrow = 1, rel_widths = module_pub_widths[top_idx])
C_bottom <- plot_grid(
  plotlist = c(lapply(module_pub_panels[bottom_idx],
                      function(p) p + theme(legend.position = "none")),
               list(ggdraw() + draw_grob(module_legend, x = 0.08, y = 0.02,
                                         width = 0.84, height = 0.96))),
  nrow = 1, rel_widths = c(module_pub_widths[bottom_idx], legend_pad)
)
C_main <- plot_grid(C_top, C_bottom, ncol = 1,
                    rel_heights = c(max(module_pub_widths[top_idx]),
                                    max(module_pub_widths[bottom_idx])))
panel_E <- C_main

panel_E <- add_overlay_tag(panel_E, "E")
options(repr.plot.width = 7.2, repr.plot.height = 4.0)
print(panel_E)

## Panel F: hard-to-distinguish cell-type marker enrichment

Also demoted from the main figure. The main figure now makes the hard-pair argument
with the LV/cell-type correlations and the ranked gene loadings; this grid is the
marker-enrichment view of the same six groups.

In [ ]:
hard_ready <- fread(snakemake@input[["hard"]])

build_hard_pub <- function(d) {
  d[, marker_row_label := abbreviate_ct(marker_row_label)]
  d[, lv_col_label := gsub("\n", " ", wrap_label(lv_col_label),
                            fixed = TRUE)]
  row_levels <- unique(d[order(marker_row_rank), marker_row_label])
  col_levels <- unique(d[order(lv_col_rank), lv_col_label])
  d[, marker_row_label := factor(marker_row_label, levels = row_levels)]
  d[, lv_col_label := factor(lv_col_label, levels = col_levels)]
  z_lim <- d$Z_LIM[1]
  q_lim <- d$Q_LIM[1]
  ggplot(d, aes(lv_col_label, marker_row_label)) +
    geom_point(aes(size = neg_log10_fdr, color = row_effect)) +
    scale_size_continuous(limits = c(0, q_lim), range = c(0.25, 1.8), guide = "none") +
    scale_color_gradientn(colors = GRID_COLORS, values = GRID_VALUES,
                          limits = c(-z_lim, z_lim), guide = "none") +
    coord_fixed() +
    labs(x = NULL, y = NULL, title = abbreviate_ct(d$panel_title[1])) +
    theme_nature_methods() +
    theme(panel.grid.major = element_line(color = "grey90", linewidth = 0.18),
          axis.line = element_blank(), axis.ticks = element_blank(),
          axis.text.x = element_text(angle = 45, hjust = 1, size = 4.3,
                                     lineheight = 0.88),
          axis.text.y = element_text(size = 4.5),
          plot.title = element_text(size = 5.1, face = "bold", hjust = 0.5,
                                    lineheight = 0.9, margin = margin(b = 0.4)),
          plot.margin = margin(0.4, 0.5, 0.4, 0.5))
}

hard_pub_panels <- lapply(split(copy(hard_ready), hard_ready$group_id), build_hard_pub)
plot_D_pub <- C_main
panel_F <- wrap_plots(hard_pub_panels, ncol = 3) &
  theme(legend.position = "none")

panel_F <- add_overlay_tag(panel_F, "F")
options(repr.plot.width = 7.2, repr.plot.height = 4.0)
print(panel_F)

## Assembly and export

In [ ]:
FIG_W   <- 183
FIG_H   <- 285
A4_W    <- 210
A4_H    <- 297
stopifnot(FIG_W <= A4_W, FIG_H <= A4_H)

# Five rows. Only A and B share one: C, D, E and F are each a 6-dataset or
# 6-group grid, and pairing any two of them collapses the tile heights to nothing.
supp1 <- wrap_plots(
  wrap_plots(panel_A, panel_B, ncol = 2, widths = c(3.2, 1)),
  panel_C,
  panel_D,
  panel_E,
  panel_F,
  ncol = 1, heights = c(0.80, 1.30, 1.25, 1.25, 1.45))

ggsave(snakemake@output[["pdf"]], supp1,
       width = FIG_W, height = FIG_H, units = "mm",
       device = cairo_pdf, bg = "white")
ggsave(snakemake@output[["svg"]], supp1,
       width = FIG_W, height = FIG_H, units = "mm",
       device = svglite::svglite, bg = "white")
ggsave(snakemake@output[["png"]], supp1,
       width = FIG_W, height = FIG_H, units = "mm",
       dpi = 600, device = ragg::agg_png, bg = "white")

cat(sprintf("supp1: %.0f x %.0f mm (%s A4 %.0f x %.0f mm)\n", FIG_W, FIG_H,
            if (FIG_W <= A4_W && FIG_H <= A4_H) "fits within" else "EXCEEDS", A4_W, A4_H))
cat("output directory:", dirname(snakemake@output[["pdf"]]), "\n")